# 09 — AHR self-play and improvement

This notebook builds the resumable Go learning loop: **AHR chooses legal moves, completed games become replay, and the learner produces candidate models.** Queen, Worker, Scout and bounded pheromone memory remain inside the board-native model from notebook 08. Notebook 11 must measure playing strength before any candidate is promoted.

**Inputs:** exported definitions from 00–06 and 08; an optional verified notebook-08 or notebook-09 model checkpoint. No lockbox data, external engine labels or upstream training cells are loaded. **Outputs:** complete checkpoints under `checkpoints/<run_id>/`, an exact-recovery validation receipt, and a validation report under `reports/`. **Default runtime:** a short recovery smoke, including a few optimizer updates on explicitly synthetic pass-game fixtures; usually under two minutes. A long 9×9 training run is explicit and opt-in.

The smoke validates mechanics, not playing strength. Incomplete games stay `UNKNOWN` and provide no terminal outcome, score or ownership labels. A terminal game means two legal passes under notebook 01's area rules; uncaptured dead stones are not removed automatically.

## Execution and recovery

Run all cells in a fresh `ahr-go` kernel. Only cells tagged `ahr-export` are imported from upstream notebooks. Definitions are reusable without executing the smoke or starting training. The final configuration cell defaults to validation.

To resume an actual run, retain the exact model, self-play and device configurations, set `AHR09_MODE=train`, `AHR09_RUN_ID` to the original run, and rerun the final training cell or Run All. `run_selfplay_training(PROJECT, run_id, model_config, config, allow_long_run=True)` loads its latest committed state automatically. A changed source/configuration, corrupt checkpoint or missing manifest fails visibly; use a new run ID for a new experiment. Checkpoints identify this resume cell and retain the latest two plus the candidate with the lowest observed training loss. That candidate is **not** an arena-approved champion.

The workspace is on the persistent boot disk, but instance-deletion policy remains unverified. Use notebook 13 or an independently verified off-instance backup before destroying the VM.

In [ ]:
from pathlib import Path
import nbformat
PROJECT = Path.cwd()
if not (PROJECT / "notebooks").is_dir():
    PROJECT = PROJECT.parent
if not (PROJECT / "notebooks/00_instance_setup_and_recovery.ipynb").is_file():
    raise RuntimeError("Run from the project workspace or its notebooks directory")
for _name in ["00_instance_setup_and_recovery.ipynb", "01_go_rules_and_independent_verifier.ipynb",
              "02_go_to_csp_and_qcsp.ipynb", "03_curricula_datasets_and_splits.ipynb",
              "04_native_ahr_architecture.ipynb", "05_csp_qcsp_pretraining.ipynb",
              "06_ahr_go_search_engine.ipynb", "08_go_specific_adaptation.ipynb"]:
    _nb = nbformat.read(PROJECT / "notebooks" / _name, as_version=4)
    for _index, _cell in enumerate(_nb.cells):
        if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
            exec(compile(_cell.source, f"{_name}:export-{_index}", "exec"), globals())
print("Imported definition cells only; no upstream training or generation was run.")

## Configuration and reproducibility

The baseline runs one game at a time and learns between games. `actor_device` can place inference on another GPU; the learner remains on `device`. This is explicit device placement, not asynchronous or distributed training. Both device choices, all RNG states, the current game's full superko history, replay order and frozen opponent snapshots are checkpointed. Search has a cooperative time budget; wall-clock cutoffs may vary across machines, so exact recovery is verified on the same device configuration with deterministic synthetic move fixtures.

Replay is bounded by positions. Search preference targets and the sampled behavior distribution are stored separately. Older AHR snapshots can act as opponents. The learner never changes a game in progress.

Frozen inference copies are cached only for the current game. They are derived from checkpointed weights and rebuilt without consuming RNG on resume. `historical_opponents` accepts explicitly pinned run/selection/SHA-256 references; those weights and provenance are embedded in the new run and retained alongside the rolling snapshot pool. No external engine labels enter training.

In [ ]:
from dataclasses import dataclass, asdict, field
import io, math, subprocess, sys, shutil, nbformat, time

SELFPLAY09_SCHEMA = "ahr_go_selfplay_v1"
SELFPLAY09_REPLAY_SCHEMA = "ahr_go_completed_replay_v1"

@dataclass(frozen=True)
class SelfPlayConfig:
    board_size: int = 9
    komi: float = 7.5
    games: int = 64
    max_plies: int = 512
    updates_per_game: int = 8
    batch_size: int = 32
    replay_capacity: int = 2048
    learning_rate: float = 0.0003
    weight_decay: float = 0.0001
    seed: int = 20260924
    device: str = "cpu"
    actor_device: str = "cpu"
    time_budget_s: float = 0.25
    temperature: float = 1.0
    exploration: float = 0.1
    opponent_probability: float = 0.25
    opponent_snapshots: int = 4
    snapshot_every_games: int = 8
    search: dict = field(default_factory=lambda: asdict(GoAHRSearchConfig()))
    purpose: str = "go_training"
    validation_force_pass: bool = False
    checkpoint_every_events: int = 32
    checkpoint_interval_s: float = 600.0
    checkpoint_every_optimizer_steps: int = 250
    historical_opponents: tuple = ()

    def __post_init__(self):
        for name in ("board_size", "games", "max_plies", "updates_per_game", "batch_size",
                     "replay_capacity", "opponent_snapshots", "snapshot_every_games",
                     "checkpoint_every_events", "checkpoint_every_optimizer_steps"):
            if type(getattr(self, name)) is not int or getattr(self, name) < 1:
                raise ValueError(name + " must be a positive integer")
        if type(self.seed) is not int or not 0 <= self.seed < 2**32:
            raise ValueError("seed must be an integer in [0, 2**32)")
        if self.board_size < 2 or self.replay_capacity < self.batch_size:
            raise ValueError("Invalid board size or replay capacity")
        for name in ("komi", "learning_rate", "weight_decay", "time_budget_s", "temperature",
                     "exploration", "opponent_probability"):
            if isinstance(getattr(self, name), bool) or not math.isfinite(getattr(self, name)):
                raise ValueError(name + " must be finite")
        if self.learning_rate <= 0 or self.weight_decay < 0 or not 0 < self.time_budget_s <= 60 or self.temperature <= 0:
            raise ValueError("Invalid optimizer, thinking-time or temperature configuration")
        if not 0 <= self.exploration <= 1 or not 0 <= self.opponent_probability <= 1:
            raise ValueError("Mixture probabilities must be in [0, 1]")
        if self.purpose not in ("go_training", "synthetic_smoke"):
            raise ValueError("Unknown run purpose")
        if self.validation_force_pass and self.purpose != "synthetic_smoke":
            raise ValueError("Scripted pass fixtures cannot be training candidates")
        if self.checkpoint_every_optimizer_steps > 250:
            raise ValueError("Checkpoint optimizer interval must be at most 250")
        if isinstance(self.checkpoint_interval_s, bool) or not math.isfinite(self.checkpoint_interval_s) or not 0 < self.checkpoint_interval_s <= 900:
            raise ValueError("Checkpoint wall interval must be in (0, 900] seconds")
        if not isinstance(self.historical_opponents, (tuple, list)):
            raise ValueError("historical_opponents must contain explicitly pinned checkpoint references")
        for reference in self.historical_opponents:
            if not isinstance(reference, dict) or set(reference) != {"run_id", "selection", "checkpoint_sha256"}:
                raise ValueError("Historical opponents require run_id, selection and checkpoint_sha256")
            if reference["selection"] not in ("latest", "best") or len(reference["checkpoint_sha256"]) != 64:
                raise ValueError("Invalid frozen opponent selection or hash")
        GoAHRSearchConfig(**self.search)
        for name in ("device", "actor_device"):
            device = torch.device(getattr(self, name))
            if device.type not in ("cpu", "cuda"):
                raise ValueError("Only CPU and CUDA backends are implemented")
            if device.type == "cuda" and (not torch.cuda.is_available() or
                    (device.index is not None and device.index >= torch.cuda.device_count())):
                raise ValueError("Requested CUDA device is unavailable")


def selfplay09_cpu_tree(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {key: selfplay09_cpu_tree(item) for key, item in value.items()}
    if isinstance(value, list):
        return [selfplay09_cpu_tree(item) for item in value]
    if isinstance(value, tuple):
        return tuple(selfplay09_cpu_tree(item) for item in value)
    return copy.deepcopy(value)


def selfplay09_source_hash(project):
    notebook = nbformat.read(Path(project) / "notebooks/09_selfplay_and_improvement.ipynb", as_version=4)
    sources = [cell.source for cell in notebook.cells if cell.cell_type == "code"
               and "ahr-export" in cell.metadata.get("tags", [])]
    return hashlib.sha256("\n\n".join(sources).encode()).hexdigest()


def selfplay09_identity(project, model_config, config):
    value = {"schema": SELFPLAY09_SCHEMA, "model_config": asdict(model_config),
             "config": asdict(config), "source_hashes": go08_source_hashes(project),
             "selfplay_source_hash": selfplay09_source_hash(project)}
    return hashlib.sha256(json.dumps(value, sort_keys=True, separators=(",", ":")).encode()).hexdigest()


def selfplay09_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.set_num_threads(1)
    if torch.cuda.is_available():
        torch.backends.cudnn.benchmark = False
        torch.backends.cuda.matmul.allow_tf32 = False


def selfplay09_clone(model_config, weights, device):
    # Constructing an inference copy must not consume the training RNG stream.
    rng = capture_rng()
    try:
        clone = GoAHR(model_config).to(device)
        clone.load_state_dict(weights, strict=True)
        return clone.eval()
    finally:
        restore_rng(rng)


def selfplay09_historical_opponents(project, model_config, config):
    """Load each pinned historical model once; embed its weights in the new run."""
    snapshots = []
    for reference in config.historical_opponents:
        model, metadata = load_go_ahr(project, reference["run_id"], device="cpu", selection=reference["selection"])
        if metadata["checkpoint_sha256"] != reference["checkpoint_sha256"]:
            raise ValueError("Historical opponent checkpoint changed: " + reference["run_id"])
        if asdict(model.config) != asdict(model_config):
            raise ValueError("Historical opponent architecture mismatch")
        snapshots.append({"model": selfplay09_cpu_tree(model.state_dict()),
                          "digest": go_model_digest(model), "optimizer_step": metadata["optimizer_step"],
                          "checkpoint_metadata": metadata, "historical": True})
    return snapshots


def selfplay09_generation_source(project, run_id, *, selection="latest", device="cpu", expected_checkpoint_sha256=None):
    """Initialize a NEW generation from a validated selected model; never resume an old run with new code."""
    model, metadata = load_go_ahr(project, run_id, device=device, selection=selection)
    if expected_checkpoint_sha256 is not None and metadata["checkpoint_sha256"] != expected_checkpoint_sha256:
        raise ValueError("Selected parent checkpoint hash changed")
    metadata = {**metadata, "handoff": "new_generation_fresh_optimizer",
                "parent_selfplay_source_hash": strict_go_checkpoint(project, run_id, selection).get("selfplay_source_hash")}
    return model, metadata


def selfplay09_cached_actor(state, components, config, runtime):
    """Derived frozen per-game inference copies; rebuilding them preserves every RNG."""
    game = state["current_game"]
    board = from_dict(game["state"])
    index = game["opponent_index"]
    uses_opponent = index is not None and board.to_play != game["learner_color"]
    key = (game["game_id"], index if uses_opponent else "learner", state["optimizer_step"])
    actors = runtime.setdefault("actors", {})
    if key not in actors:
        weights = state["opponents"][index]["model"] if uses_opponent else components[0].state_dict()
        actor = selfplay09_clone(GoAHRConfig(**state["model_config"]), weights, config.actor_device)
        actors[key] = (actor, go_model_digest(actor))
    return actors[key]


def selfplay09_checkpoint_due(state, before, config, last_commit_event, last_commit_update, last_commit_time, *, improved=False):
    """Commit phase/epoch boundaries, every best, bounded events/updates/time, and final state."""
    return (improved or state["phase"] != before["phase"] or
            state["sampler"]["epoch"] != before["epoch"] or
            state["global_step"] - last_commit_event >= config.checkpoint_every_events or
            state["optimizer_step"] - last_commit_update >= config.checkpoint_every_optimizer_steps or
            time.monotonic() - last_commit_time >= config.checkpoint_interval_s or
            state["phase"] == "complete")


## Verify games before deriving learning targets

Only two-pass terminal games enter replay. Every transition is checked against both notebook 01 and notebook 06's independently written one-move verifier. The final board supplies area score and ownership; perspective is always the player to move in the stored position. A ply cap produces `UNKNOWN`, never a loss. Policy targets describe the AHR procedure that generated them and are estimates, not proofs of optimal moves. Deadline-only pass fallbacks receive zero policy weight; their fallback provenance is preserved. A genuinely completed game still supplies its verified terminal targets.

In [ ]:
def selfplay09_policy(state, values):
    probabilities = np.asarray(values, dtype=np.float64)
    if probabilities.shape != (state.pass_action + 1,) or not np.isfinite(probabilities).all() or (probabilities < 0).any():
        raise ValueError("Invalid policy distribution")
    allowed = np.zeros_like(probabilities, dtype=bool)
    allowed[list(legal_actions(state))] = True
    if probabilities[~allowed].sum() > 1e-7 or probabilities.sum() <= 0:
        raise ValueError("Policy must assign mass only to legal moves")
    probabilities[~allowed] = 0
    return probabilities / probabilities.sum()


def selfplay09_verify_completed(game):
    state = from_dict(game["initial_state"])
    if not game["records"]:
        raise ValueError("A completed replay game must contain a trajectory")
    for record in game["records"]:
        if record["state"] != state.to_dict():
            raise ValueError("Game trajectory or superko history changed")
        policy = selfplay09_policy(state, record["policy_target"])
        behavior = selfplay09_policy(state, record["behavior_policy"])
        action = record["action"]
        if type(action) is not int or not 0 <= action < len(policy) or behavior[action] <= 0:
            raise ValueError("Played action is absent from the behavior distribution")
        child = play(state, action)
        if not verify_go_emission(state, action, child):
            raise ValueError("Independent legal-move verifier disagreed")
        state = child
    if not state.terminal or state.to_dict() != game["state"]:
        raise ValueError("No independently verified two-pass terminal game")
    return state


def selfplay09_completed_examples(game):
    final = selfplay09_verify_completed(game)
    examples = []
    for record in game["records"]:
        state = from_dict(record["state"])
        targets = terminal_go_targets(final, state.to_play)
        examples.append({"schema": SELFPLAY09_REPLAY_SCHEMA,
                         "state": copy.deepcopy(record["state"]),
                         "policy_target": list(record["policy_target"]),
                         "policy_weight": 0.0 if record.get("fallback_pass", False) else 1.0,
                         **targets, "game_id": game["game_id"],
                         "source": {"procedure": "ahr_search_preferences_with_exploration",
                                    "model_digest": record["model_digest"],
                                    "behavior_policy": list(record["behavior_policy"]),
                                    "action": record["action"], "search": copy.deepcopy(record["search"]),
                                    "temperature": record["temperature"], "exploration": record["exploration"],
                                    "policy_target_kind": record["policy_target_kind"],
                                    "fallback_pass": record.get("fallback_pass", False),
                                    "purpose": game["purpose"], "terminal_status": "VERIFIED_TERMINAL"}})
    return examples


def selfplay09_choose(state, model, config):
    if config.validation_force_pass:
        policy = np.zeros(state.pass_action + 1, dtype=np.float64)
        policy[state.pass_action] = 1.0
        return {"action": state.pass_action, "policy_target": policy.tolist(),
                "behavior_policy": policy.tolist(), "status": "SYNTHETIC_FIXTURE",
                "strategic_status": "UNKNOWN", "reason": "scripted_pass_recovery_fixture",
                "elapsed_s": 0.0, "nodes": 0, "model_calls": 0,
                "policy_target_kind": "synthetic_pass_fixture", "fallback_pass": False}
    result = choose_go_ahr_move(state, model, time_budget_s=config.time_budget_s,
                               search_config=GoAHRSearchConfig(**config.search))
    policy = selfplay09_policy(state, result["policy_target"])
    positive = policy > 0
    behavior = np.zeros_like(policy)
    log_weights = np.log(policy[positive]) / config.temperature
    behavior[positive] = np.exp(log_weights - log_weights.max())
    behavior /= behavior.sum()
    uniform = np.zeros_like(policy)
    legal = list(legal_actions(state))
    uniform[legal] = 1 / len(legal)
    behavior = (1 - config.exploration) * behavior + config.exploration * uniform
    action = int(np.random.choice(len(policy), p=behavior))
    return {**result, "action": action, "policy_target": policy.tolist(),
            "behavior_policy": behavior.tolist(),
            "policy_target_kind": result.get("policy_target_kind", "ahr_search_preferences"),
            "fallback_pass": bool(result.get("fallback_pass", False))}


## Bounded commits with exact recovery

A state-machine event starts a game, plays a move, admits a verified terminal game, or performs one learner update. Full interruption-safe checkpoints commit at each phase/stage and replay-epoch boundary, each best training loss, every 32 events, every 600 seconds, or 250 optimizer steps, whichever comes first. Event processing is sequential; cadence is checked after each bounded event. Every clean bounded-call return also commits. `global_step` counts events; `optimizer_step` counts parameter updates.

An unexpected process kill discards only work since the last valid commit; resume restores that commit and redoes those events. All models, optimizer/scheduler/scaler, RNG, replay, sampler cursor, full superko history and opponent snapshots remain in each checkpoint. The latest two valid checkpoints plus best are retained. A corrupt manifest fails visibly. The best checkpoint is a training-loss candidate; notebook 11 controls promotion.

In [ ]:
def selfplay09_read_checkpoint(run_dir, expected_identity=None):
    run_dir = Path(run_dir)
    manifest_path = run_dir / "manifest.json"
    if not manifest_path.exists():
        if list(run_dir.glob("*.pt")):
            raise RuntimeError("Checkpoint payload exists without its committed manifest")
        return None
    try:
        manifest = json.loads(manifest_path.read_text())
        entries = manifest["entries"]
        if not entries or len({item["file"] for item in entries}) != len(entries):
            raise ValueError("Invalid checkpoint entries")
        if manifest["latest"] != max(entries, key=lambda item: item["global_step"])["file"]:
            raise ValueError("Manifest latest does not identify the newest event")
        if manifest.get("best") not in {item["file"] for item in entries}:
            raise ValueError("Manifest best entry is missing")
        loaded = {}
        for entry in entries:
            state = _checkpoint_entry_state(run_dir, entry)
            if state is None:
                raise ValueError("Missing, corrupt or inconsistent checkpoint: " + entry["file"])
            if state.get("schema") != SELFPLAY09_SCHEMA:
                raise ValueError("Unexpected self-play checkpoint schema")
            if expected_identity is not None and state["identity"] != expected_identity:
                raise ValueError("Source, model, device or run configuration changed")
            if state["optimizer_step"] < 0 or state["global_step"] < state["optimizer_step"]:
                raise ValueError("Invalid event or optimizer cursor")
            loaded[entry["file"]] = state
        return loaded[manifest["latest"]]
    except Exception as error:
        raise RuntimeError("Strict self-play recovery refused: " + str(error)) from error


def selfplay09_components(model_config, config, initial_weights=None):
    selfplay09_seed(config.seed)
    model = GoAHR(model_config).to(config.device)
    if initial_weights is not None:
        model.load_state_dict(initial_weights, strict=True)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.learning_rate,
                                  weight_decay=config.weight_decay)
    scheduler = torch.optim.lr_scheduler.ExponentialLR(optimizer, gamma=0.999)
    # Full precision is the baseline; scaler is explicit and checkpointed even while disabled.
    scaler = torch.amp.GradScaler("cuda", enabled=False)
    return model, optimizer, scheduler, scaler


def selfplay09_new_state(project, model_config, config, components):
    model, optimizer, scheduler, scaler = components
    weights = selfplay09_cpu_tree(model.state_dict())
    historical = selfplay09_historical_opponents(project, model_config, config)
    return {"schema": SELFPLAY09_SCHEMA, "identity": selfplay09_identity(project, model_config, config),
            "model_config": asdict(model_config), "config": asdict(config),
            "source_hashes": go08_source_hashes(project), "selfplay_source_hash": selfplay09_source_hash(project),
            "provenance": {"training_route": "go_selfplay", "trained_on_go": True,
                           "purpose": config.purpose, "candidate_only": True,
                           "best_selection": "lowest_observed_training_loss_not_arena_strength"},
            "model": weights, "initial_model": copy.deepcopy(weights),
            "optimizer": selfplay09_cpu_tree(optimizer.state_dict()),
            "scheduler": scheduler.state_dict(), "scaler": scaler.state_dict(), "rng": capture_rng(),
            "global_step": 0, "optimizer_step": 0, "phase": "new_game", "current_game": None,
            "replay": [], "replay_version": 0,
            "sampler": {"permutation": [], "position": 0, "epoch": 0, "replay_version": -1},
            "games_finished": 0, "updates_remaining": 0,
            "opponents": historical + [{"model": copy.deepcopy(weights), "digest": go_model_digest(model), "optimizer_step": 0}],
            "metrics": {"completed_games": 0, "unknown_games": 0, "verified_plies": 0,
                        "actual_ahr_plies": 0, "search_model_calls": 0, "search_deadline_exceedances": 0,
                        "examples_admitted": 0, "best_training_loss": None, "loss_history": [],
                        "game_history": [], "last_update": None},
            "distributed": {"world_size": 1, "learner_device": config.device,
                            "actor_device": config.actor_device, "asynchronous_actors": False}}


def selfplay09_pack(state, components):
    model, optimizer, scheduler, scaler = components
    state["model"] = selfplay09_cpu_tree(model.state_dict())
    state["optimizer"] = selfplay09_cpu_tree(optimizer.state_dict())
    state["scheduler"] = copy.deepcopy(scheduler.state_dict())
    state["scaler"] = copy.deepcopy(scaler.state_dict())
    state["rng"] = capture_rng()
    return state


def selfplay09_restore(state, components):
    model, optimizer, scheduler, scaler = components
    model.load_state_dict(state["model"], strict=True)
    optimizer.load_state_dict(state["optimizer"])
    scheduler.load_state_dict(state["scheduler"])
    scaler.load_state_dict(state["scaler"])
    restore_rng(state["rng"])


def selfplay09_sample(state, config):
    replay, sampler = state["replay"], state["sampler"]
    if not replay:
        raise ValueError("Cannot learn without verified completed replay")
    if sampler["replay_version"] != state["replay_version"]:
        sampler.update(permutation=np.random.permutation(len(replay)).tolist(), position=0,
                       replay_version=state["replay_version"])
    selected = []
    for _ in range(min(config.batch_size, len(replay))):
        if sampler["position"] >= len(sampler["permutation"]):
            sampler.update(permutation=np.random.permutation(len(replay)).tolist(), position=0,
                           epoch=sampler["epoch"] + 1)
        selected.append(sampler["permutation"][sampler["position"]])
        sampler["position"] += 1
    return [replay[index] for index in selected]

### Advance one game or learner event

In [ ]:
def selfplay09_event(state, components, config, runtime=None):
    model, optimizer, scheduler, scaler = components
    improved = False
    runtime = {} if runtime is None else runtime
    if state["phase"] == "new_game":
        runtime.pop("actors", None)
        if state["games_finished"] >= config.games:
            state["phase"] = "complete"
        else:
            opponent = None
            if state["opponents"] and random.random() < config.opponent_probability:
                opponent = random.randrange(len(state["opponents"]))
            board = initial_state(config.board_size, config.komi)
            state["current_game"] = {"game_id": state["games_finished"], "purpose": config.purpose,
                                     "initial_state": board.to_dict(), "state": board.to_dict(),
                                     "records": [], "status": "IN_PROGRESS", "reason": None,
                                     "opponent_index": opponent,
                                     "learner_color": 1 if state["games_finished"] % 2 == 0 else -1,
                                     "learner_digest": go_model_digest(model)}
            state["phase"] = "play"
    elif state["phase"] == "play":
        game = state["current_game"]
        board = from_dict(game["state"])
        if board.terminal:
            state["phase"] = "ingest"
        elif len(game["records"]) >= config.max_plies:
            game.update(status="UNKNOWN", reason="max_plies_cutoff")
            state["phase"] = "ingest"
        else:
            index = game["opponent_index"]
            uses_opponent = index is not None and board.to_play != game["learner_color"]
            actor, actor_digest = selfplay09_cached_actor(state, components, config, runtime)
            result = selfplay09_choose(board, actor, config)
            child = play(board, result["action"])
            if not verify_go_emission(board, result["action"], child):
                raise RuntimeError("Independent verifier rejected the proposed self-play move")
            game["records"].append({"state": board.to_dict(), "action": result["action"],
                                    "policy_target": result["policy_target"],
                                    "behavior_policy": result["behavior_policy"],
                                    "model_digest": actor_digest,
                                    "temperature": config.temperature, "exploration": config.exploration,
                                    "policy_target_kind": result["policy_target_kind"],
                                    "fallback_pass": result.get("fallback_pass", False),
                                    "search": {"config": copy.deepcopy(config.search),
                                               "time_budget_s": config.time_budget_s,
                                               "status": result["status"], "strategic_status": "UNKNOWN",
                                               "reason": result.get("reason"), "nodes": result.get("nodes", 0),
                                               "model_calls": result.get("model_calls", 0),
                                               "elapsed_s": float(result.get("elapsed_s", 0.0)),
                                               "cooperative_deadline_exceeded": float(result.get("elapsed_s", 0.0)) > config.time_budget_s}})
            game["state"] = child.to_dict()
            state["metrics"]["verified_plies"] += 1
            state["metrics"]["actual_ahr_plies"] += int(not config.validation_force_pass)
            state["metrics"]["search_model_calls"] += int(result.get("model_calls", 0))
            state["metrics"]["search_deadline_exceedances"] += int(float(result.get("elapsed_s", 0.0)) > config.time_budget_s)
            if child.terminal:
                game.update(status="VERIFIED_TERMINAL", reason="two_legal_passes")
                state["phase"] = "ingest"
    elif state["phase"] == "ingest":
        game = state["current_game"]
        if game["status"] == "VERIFIED_TERMINAL":
            examples = selfplay09_completed_examples(game)
            state["replay"] = (state["replay"] + examples)[-config.replay_capacity:]
            state["replay_version"] += 1
            state["metrics"]["completed_games"] += 1
            state["metrics"]["examples_admitted"] += len(examples)
        elif game["status"] == "UNKNOWN":
            state["metrics"]["unknown_games"] += 1
        else:
            raise RuntimeError("Only verified terminal or explicitly UNKNOWN games can close")
        state["metrics"]["game_history"].append({"game_id": game["game_id"], "status": game["status"],
                                                "reason": game["reason"], "plies": len(game["records"]),
                                                "admitted": game["status"] == "VERIFIED_TERMINAL"})
        state["games_finished"] += 1
        state["updates_remaining"] = config.updates_per_game if state["replay"] else 0
        state["current_game"] = None
        state["phase"] = "learn" if state["updates_remaining"] else "new_game"
    elif state["phase"] == "learn":
        examples = selfplay09_sample(state, config)
        model.train()
        optimizer.zero_grad(set_to_none=True)
        loss, update_metrics = go_learning_loss(model, examples)
        if not torch.isfinite(loss):
            raise RuntimeError("Nonfinite self-play training loss")
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        gradient_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), 5.0, error_if_nonfinite=True)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        state["optimizer_step"] += 1
        state["updates_remaining"] -= 1
        loss_value = float(loss.detach().cpu())
        metrics = {key: float(value.detach().cpu()) if torch.is_tensor(value) else float(value)
                   for key, value in update_metrics.items() if isinstance(value, (int, float)) or torch.is_tensor(value)}
        metrics.update(loss=loss_value, gradient_norm=float(gradient_norm.detach().cpu()),
                       optimizer_step=state["optimizer_step"], replay_positions=len(state["replay"]))
        state["metrics"]["last_update"] = metrics
        state["metrics"]["loss_history"] = (state["metrics"]["loss_history"] + [metrics])[-1000:]
        best = state["metrics"]["best_training_loss"]
        if best is None or loss_value < best:
            state["metrics"]["best_training_loss"] = loss_value
            improved = True
        if state["updates_remaining"] == 0:
            if state["games_finished"] % config.snapshot_every_games == 0:
                snapshot = {"model": selfplay09_cpu_tree(model.state_dict()),
                            "digest": go_model_digest(model), "optimizer_step": state["optimizer_step"]}
                historical = [item for item in state["opponents"] if item.get("historical", False)]
                rolling = [item for item in state["opponents"] if not item.get("historical", False)]
                state["opponents"] = historical + (rolling + [snapshot])[-config.opponent_snapshots:]
            state["phase"] = "new_game"
    elif state["phase"] != "complete":
        raise RuntimeError("Unknown self-play phase")
    state["global_step"] += 1
    return improved

In [ ]:
def selfplay09_receipt_key(project, model_config, config):
    identity = {"model_config": asdict(model_config), "device": config.device,
                "actor_device": config.actor_device, "source_hashes": go08_source_hashes(project),
                "selfplay_source_hash": selfplay09_source_hash(project),
                "checkpoint_policy": {name: getattr(config, name) for name in
                    ("checkpoint_every_events", "checkpoint_interval_s", "checkpoint_every_optimizer_steps")}}
    return hashlib.sha256(json.dumps(identity, sort_keys=True).encode()).hexdigest()


def run_selfplay_training(project, run_id, model_config, config, *, initial_model=None,
                          max_events=None, allow_long_run=False, initial_metadata=None):
    """Sequential source-bound training; immutable full checkpoints with bounded cadence.

    An unexpected kill replays uncommitted events from the last valid commit. A clean
    max_events exit always commits exactly at that event. Derived actor caches are
    rebuilt from committed weights without consuming the restored RNG stream.
    """
    project = Path(project).resolve()
    if not isinstance(run_id, str) or not run_id or Path(run_id).name != run_id or run_id in (".", ".."):
        raise ValueError("Use a simple nonempty run ID")
    if model_config.board_size != config.board_size:
        raise ValueError("Model and self-play board sizes must agree")
    if max_events is not None and (type(max_events) is not int or max_events < 0):
        raise ValueError("max_events is an optional nonnegative event limit")
    # Conservative upper bound assumes every allowed ply consumes its thinking budget.
    long_run = (config.games * config.max_plies * config.time_budget_s > 900 or
                config.games * config.updates_per_game > 250)
    if long_run and config.purpose != "synthetic_smoke":
        if not allow_long_run:
            raise RuntimeError("Long self-play run requires allow_long_run=True after an exact recovery smoke")
        key = selfplay09_receipt_key(project, model_config, config)
        path = project / "reports/09_recovery_receipts" / (key + ".json")
        if not path.is_file():
            raise RuntimeError("Run validate_selfplay_recovery with this model and device configuration first")
        receipt = json.loads(path.read_text())
        if receipt.get("key") != key or not receipt.get("exact_resume") or not receipt.get("fresh_process"):
            raise RuntimeError("Exact recovery receipt is invalid")
    if initial_metadata is not None and initial_model is None:
        raise ValueError("Initial checkpoint metadata requires its loaded model")
    initial_weights = None
    if initial_model is not None:
        if asdict(initial_model.config) != asdict(model_config):
            raise ValueError("Initial checkpoint model configuration mismatch")
        initial_weights = selfplay09_cpu_tree(initial_model.state_dict())
        if initial_metadata is not None and initial_metadata.get("model_sha256") != go_model_digest(initial_model):
            raise ValueError("Initial checkpoint metadata and model digest disagree")
    run_dir = project / "checkpoints" / run_id
    identity = selfplay09_identity(project, model_config, config)
    with checkpoint_lock(run_dir):
        state = selfplay09_read_checkpoint(run_dir, identity)
        components = selfplay09_components(model_config, config, initial_weights)
        if state is None:
            state = selfplay09_new_state(project, model_config, config, components)
            state["initial_model_digest"] = go_model_digest(components[0])
            state["initialization"] = {"source": "checkpoint" if initial_metadata else "provided_model" if initial_model is not None else "random_seed",
                                       "seed": config.seed, "model_digest": state["initial_model_digest"],
                                       "checkpoint_metadata": copy.deepcopy(initial_metadata)}
            _save_checkpoint_unlocked(run_dir, selfplay09_pack(state, components), best=True,
                                      resume_cell="09: Run or resume explicit self-play training")
        else:
            if initial_model is not None and go_model_digest(initial_model) != state["initial_model_digest"]:
                raise RuntimeError("Requested initialization differs from the committed run")
            selfplay09_restore(state, components)
        first_event = last_commit_event = state["global_step"]
        last_commit_update = state["optimizer_step"]
        last_commit_time = time.monotonic()
        runtime = {}
        while state["phase"] != "complete":
            if max_events is not None and state["global_step"] - first_event >= max_events:
                break
            before = {"phase": state["phase"], "epoch": state["sampler"]["epoch"]}
            improved = selfplay09_event(state, components, config, runtime=runtime)
            if selfplay09_checkpoint_due(state, before, config, last_commit_event, last_commit_update,
                                         last_commit_time, improved=improved):
                _save_checkpoint_unlocked(run_dir, selfplay09_pack(state, components), best=improved,
                                          resume_cell="09: Run or resume explicit self-play training")
                last_commit_event, last_commit_update = state["global_step"], state["optimizer_step"]
                last_commit_time = time.monotonic()
        # A bounded call is a deliberate interruption boundary, never an unsaved return.
        if state["global_step"] != last_commit_event:
            _save_checkpoint_unlocked(run_dir, selfplay09_pack(state, components),
                                      resume_cell="09: Run or resume explicit self-play training")
        return selfplay09_cpu_tree(state)

### Launch fresh notebook-only workers

In [ ]:
def selfplay09_bootstrap():
    return """import sys,json,nbformat
from pathlib import Path
project=Path(sys.argv[1]);request=json.loads(Path(sys.argv[2]).read_text())
namespace={'__name__':'__ahr_notebook_worker__','__file__':str(project/'notebooks/09_selfplay_and_improvement.ipynb')}
for name in ['00_instance_setup_and_recovery.ipynb','01_go_rules_and_independent_verifier.ipynb','02_go_to_csp_and_qcsp.ipynb','03_curricula_datasets_and_splits.ipynb','04_native_ahr_architecture.ipynb','05_csp_qcsp_pretraining.ipynb','06_ahr_go_search_engine.ipynb','08_go_specific_adaptation.ipynb','09_selfplay_and_improvement.ipynb']:
    notebook=nbformat.read(project/'notebooks'/name,as_version=4)
    for cell in notebook.cells:
        if cell.cell_type=='code' and 'ahr-export' in cell.metadata.get('tags',[]):exec(compile(cell.source,name,'exec'),namespace)
config=namespace['SelfPlayConfig'](**request['config'])
model_config=namespace['GoAHRConfig'](**request['model_config'])
state=namespace['run_selfplay_training'](project,request['run_id'],model_config,config,max_events=request.get('max_events'))
print(json.dumps({'phase':state['phase'],'event':state['global_step'],'optimizer_step':state['optimizer_step'],'completed_games':state['metrics']['completed_games']}))
"""

### Compare complete recovery states

In [ ]:
def validate_selfplay_recovery(project, model_config, *, device="cpu", actor_device=None, production_config=None):
    """Three fresh processes: uninterrupted, interrupted mid-game, and exact resume."""
    project = Path(project).resolve()
    actor_device = device if actor_device is None else actor_device
    validated_sources = go08_source_hashes(project)
    validated_selfplay_source = selfplay09_source_hash(project)
    config = SelfPlayConfig(board_size=model_config.board_size, games=2, max_plies=4,
                            updates_per_game=2, batch_size=2, replay_capacity=8,
                            device=device, actor_device=actor_device, purpose="synthetic_smoke",
                            validation_force_pass=True, opponent_probability=1.0,
                            snapshot_every_games=1, search=asdict(GoAHRSearchConfig(max_nodes=4, max_depth=1, branch_width=2)))
    if production_config is not None:
        if production_config.device != device or production_config.actor_device != actor_device:
            raise ValueError("Production validation device selection disagrees")
        config = SelfPlayConfig(**{**asdict(production_config), "games": 2, "max_plies": 4,
                                  "updates_per_game": 2, "purpose": "synthetic_smoke",
                                  "validation_force_pass": True, "opponent_probability": 1.0,
                                  "snapshot_every_games": 1})
    token = uuid.uuid4().hex[:10]
    prefix = "selfplay09-smoke-" + str(device).replace(":", "-") + "-" + token
    full_run, split_run, update_run = prefix + "-full", prefix + "-resume", prefix + "-update-resume"
    requests_dir = project / "reports/09_recovery_requests" / token
    requests_dir.mkdir(parents=True, exist_ok=True)
    outputs = []
    for index, (run_id, event_limit) in enumerate([(full_run, None), (split_run, 2), (split_run, None), (update_run, 5), (update_run, None)]):
        request = {"run_id": run_id, "model_config": asdict(model_config), "config": asdict(config),
                   "max_events": event_limit}
        path = requests_dir / (str(index) + ".json")
        atomic_json(path, request)
        process = subprocess.run([sys.executable, "-c", selfplay09_bootstrap(), str(project), str(path)],
                                 cwd=project, capture_output=True, text=True, timeout=300)
        atomic_write_bytes(requests_dir / (str(index) + ".log"), (process.stdout + process.stderr).encode())
        if process.returncode:
            raise RuntimeError("Fresh-process recovery validation failed: " + process.stderr[-4000:])
        outputs.append(json.loads(process.stdout.strip().splitlines()[-1]))
    identity = selfplay09_identity(project, model_config, config)
    full = selfplay09_read_checkpoint(project / "checkpoints" / full_run, identity)
    resumed = selfplay09_read_checkpoint(project / "checkpoints" / split_run, identity)
    if not architecture_equal(full, resumed):
        unequal = [key for key in full if not architecture_equal(full[key], resumed[key])]
        raise AssertionError("Exact self-play recovery diverged in " + str(unequal))
    update_resumed = selfplay09_read_checkpoint(project / "checkpoints" / update_run, identity)
    if not architecture_equal(full, update_resumed):
        unequal = [key for key in full if not architecture_equal(full[key], update_resumed[key])]
        raise AssertionError("Update-boundary exact recovery diverged in " + str(unequal))
    if outputs[3]["event"] != 5 or outputs[3]["optimizer_step"] != 1:
        raise AssertionError("Update interruption did not occur after one optimizer step")
    if outputs[1]["event"] != 2 or outputs[1]["optimizer_step"] != 0:
        raise AssertionError("Interruption did not occur after the first game ply")
    if full["optimizer_step"] != 4 or full["metrics"]["completed_games"] != 2:
        raise AssertionError("Recovery smoke did not cover completed games and actual updates")
    if architecture_equal(full["model"], full["initial_model"]):
        raise AssertionError("Optimizer smoke did not change model parameters")
    # Exercise UNKNOWN exclusion using the same verified fixture, cut off before two passes.
    cutoff_config = SelfPlayConfig(**{**asdict(config), "games": 1, "max_plies": 1})
    cutoff = run_selfplay_training(project, prefix + "-unknown", model_config, cutoff_config)
    if cutoff["metrics"]["unknown_games"] != 1 or cutoff["replay"] or cutoff["optimizer_step"]:
        raise AssertionError("Cutoff game leaked into terminal supervision")
    # Exercise the actual board-native AHR search route as well as deterministic fixtures.
    search_config = SelfPlayConfig(**{**asdict(config), "games": 1, "max_plies": 2,
                                     "validation_force_pass": False, "time_budget_s": 1.0})
    search_smoke = run_selfplay_training(project, prefix + "-search", model_config, search_config)
    if search_smoke["metrics"]["actual_ahr_plies"] != 2 or search_smoke["metrics"]["search_model_calls"] < 1:
        raise AssertionError("Actual AHR search route was not exercised")
    # A deadline-only pass fallback must never become a positive policy label.
    fallback_config = SelfPlayConfig(**{**asdict(config), "games": 1, "max_plies": 4,
                                       "validation_force_pass": False, "time_budget_s": 1e-12,
                                       "exploration": 0.0})
    fallback_components = selfplay09_components(model_config, fallback_config)
    fallback_state = selfplay09_new_state(project, model_config, fallback_config, fallback_components)
    for _ in range(3):
        selfplay09_event(fallback_state, fallback_components, fallback_config)
    fallback_examples = selfplay09_completed_examples(fallback_state["current_game"])
    if not all(e["policy_weight"] == 0 and e["source"]["fallback_pass"] and e["terminal_targets_verified"]
               for e in fallback_examples):
        raise AssertionError("Deadline fallback leaked into positive policy supervision")
    # Tampered manifest hashes and orphaned payloads must never silently recover.
    failure_checks = []
    with tempfile.TemporaryDirectory(prefix="selfplay09-invalid-", dir=project / "checkpoints") as temporary:
        fixture = Path(temporary)
        source_dir = project / "checkpoints" / split_run
        shutil.copytree(source_dir, fixture / "bad_hash")
        manifest_path = fixture / "bad_hash/manifest.json"
        manifest = json.loads(manifest_path.read_text())
        manifest["entries"][-1]["sha256"] = "0" * 64
        atomic_json(manifest_path, manifest)
        for label, directory in [("bad_hash", fixture / "bad_hash")]:
            try:
                selfplay09_read_checkpoint(directory, identity)
                raise AssertionError("Invalid checkpoint unexpectedly loaded")
            except RuntimeError:
                failure_checks.append(label)
        (fixture / "orphan").mkdir()
        (fixture / "orphan/orphan.pt").write_bytes(b"not a committed checkpoint")
        try:
            selfplay09_read_checkpoint(fixture / "orphan", identity)
            raise AssertionError("Orphaned checkpoint unexpectedly restarted")
        except RuntimeError:
            failure_checks.append("missing_manifest")
        try:
            selfplay09_read_checkpoint(source_dir, "wrong-identity")
            raise AssertionError("Source/configuration mismatch unexpectedly resumed")
        except RuntimeError:
            failure_checks.append("identity_mismatch")
    manifest = json.loads((project / "checkpoints" / split_run / "manifest.json").read_text())
    latest_steps = sorted(entry["global_step"] for entry in manifest["entries"])
    if len(latest_steps) < 2 or latest_steps[-1] != full["global_step"]:
        raise AssertionError("Latest two valid resumable checkpoints were not retained")
    if go08_source_hashes(project) != validated_sources or selfplay09_source_hash(project) != validated_selfplay_source:
        raise RuntimeError("Notebook exports changed during recovery validation; rerun against frozen sources")
    key = selfplay09_receipt_key(project, model_config, config)
    report = {"schema": "ahr_selfplay_recovery_receipt_v1", "key": key,
              "exact_resume": True, "fresh_process": True, "update_boundary_exact": True, "device": device,
              "actor_device": actor_device, "model_config": asdict(model_config),
              "optimizer_steps": full["optimizer_step"], "completed_synthetic_games": 2,
              "interrupted_after_event": 2, "interrupted_after_update": 1, "final_event": full["global_step"],
              "tested_config": asdict(config), "production_config": None if production_config is None else asdict(production_config),
              "unknown_excluded": True, "fallback_policy_excluded": True, "failure_checks": failure_checks,
              "checkpoint_runs": [full_run, split_run, update_run, prefix + "-unknown", prefix + "-search"],
              "actual_search_plies": search_smoke["metrics"]["actual_ahr_plies"],
              "actual_search_model_calls": search_smoke["metrics"]["search_model_calls"],
              "retained_events": latest_steps, "source_hashes": go08_source_hashes(project),
              "selfplay_source_hash": selfplay09_source_hash(project),
              "strength_claim": "NONE: synthetic recovery fixtures only"}
    atomic_json(project / "reports/09_recovery_receipts" / (key + ".json"), report)
    return report

## Validate recovery in fresh processes

The recovery smoke compares an uninterrupted run with two fresh-process interruptions: after the first game ply and after the first optimizer update. It compares **every saved field** exactly, including model tensors, optimizer moments, RNG, replay, sampler, opponents and metrics. Four optimizer updates use two verified scripted pass games. UNKNOWN exclusion, deadline-fallback policy exclusion, actual AHR search and strict corrupt/source-mismatch recovery checks also run.

`validate_selfplay_recovery(..., production_config=config)` tests the selected architecture, devices, optimizer, batch/replay limits, search settings and checkpoint cadence. Only game count/ply limit/update count and scripted move choice are shortened for the disposable smoke. Wall-clock-budgeted production search can branch differently when hardware timing changes; exact recovery does not imply reproducible wall-clock cutoffs.

In [ ]:
SMOKE_MODEL_CONFIG = GoAHRConfig(board_size=9, spatial_channels=8, hidden_dim=16,
                                 queen_cycles=1, worker_steps=1, scout_steps=1, dropout=0.1)
RECOVERY_REPORTS = [validate_selfplay_recovery(PROJECT, SMOKE_MODEL_CONFIG, device="cpu")]
if torch.cuda.is_available():
    gpu_actor = "cuda:1" if torch.cuda.device_count() > 1 else "cuda:0"
    RECOVERY_REPORTS.append(validate_selfplay_recovery(PROJECT, SMOKE_MODEL_CONFIG,
                                                      device="cuda:0", actor_device=gpu_actor))
print(json.dumps([{key: report[key] for key in ("device", "actor_device", "exact_resume", "fresh_process",
                                               "optimizer_steps", "completed_synthetic_games", "unknown_excluded")}
                  for report in RECOVERY_REPORTS], indent=2))

## Verify the warmup-to-self-play handoff

If the completed `go08-initial-warmup-v2` checkpoint is present, this check opens a new real-training run at event zero using its **best** model, verifies every initial weight and a fresh optimizer, then opens the same run again to verify exact resume. It performs no self-play moves or parameter updates. If the warmup artifact is absent, this optional integration check is recorded as unavailable; the self-contained recovery tests above still run.

In [ ]:
HANDOFF09 = {"status": "NOT_RUN_NO_WARMUP_CHECKPOINT"}
warmup_run = os.environ.get("AHR09_WARMUP_VALIDATION_RUN", "go08-initial-warmup-v2")
if Path(warmup_run).name != warmup_run or warmup_run in (".", ".."):
    raise ValueError("Use a simple warmup validation run ID")
warmup_manifest = PROJECT / "checkpoints" / warmup_run / "manifest.json"
if warmup_manifest.is_file():
    warmup_model, warmup_metadata = load_go_ahr(PROJECT, warmup_run, device="cpu", selection="best")
    handoff_config = SelfPlayConfig(games=1, max_plies=2, updates_per_game=1, batch_size=2,
                                   replay_capacity=8, device="cpu", actor_device="cpu")
    handoff_run = "selfplay09-warmup-handoff-" + uuid.uuid4().hex[:10]
    handoff = run_selfplay_training(PROJECT, handoff_run, warmup_model.config, handoff_config,
                                   initial_model=warmup_model, initial_metadata=warmup_metadata, max_events=0)
    assert architecture_equal(handoff["model"], selfplay09_cpu_tree(warmup_model.state_dict()))
    assert handoff["optimizer_step"] == 0 and handoff["optimizer"]["state"] == {}
    assert handoff["initial_model_digest"] == warmup_metadata["model_sha256"]
    assert handoff["initialization"]["checkpoint_metadata"]["checkpoint_sha256"] == warmup_metadata["checkpoint_sha256"]
    handoff_resumed = run_selfplay_training(PROJECT, handoff_run, warmup_model.config, handoff_config, max_events=0)
    assert architecture_equal(handoff, handoff_resumed)
    HANDOFF09 = {"status": "PASS", "source_run": warmup_run, "selection": "best",
                 "source_optimizer_steps": warmup_metadata["optimizer_step"], "run_id": handoff_run,
                 "initial_model_sha256": handoff["initial_model_digest"], "optimizer_fresh": True,
                 "selfplay_optimizer_steps": 0, "exact_event_zero_resume": True}
print(json.dumps(HANDOFF09, indent=2))

## Run or resume explicit self-play training

The shipped default stops after validation. To start a full run, select a reviewed initial Go model checkpoint from notebook 08 or a previous notebook09 generation; random initialization requires the explicit `AHR09_INITIALIZATION=random` option. Set the environment variables below before launching Jupyter or replace the visible values in this cell:

- `AHR09_MODE=train` selects the training branch.
- `AHR09_RUN_ID=go-selfplay-09-v1` identifies the run and its exact resume state.
- `AHR09_INITIALIZATION=checkpoint` is the default for a fresh training run; `AHR09_INITIAL_RUN=<notebook-08-or-09-run-id>` selects the parent checkpoint. `AHR09_INITIAL_SELECTION=latest` selects its latest model by default; use `best` only deliberately. A new run starts a fresh optimizer and preserves parent provenance. A missing or invalid selected checkpoint fails visibly.
- `AHR09_INITIALIZATION=random` explicitly selects a random baseline instead. Its seed and initial weights are preserved.
- `AHR09_DEVICE=cuda:0` and `AHR09_ACTOR_DEVICE=cuda:1` select learner and actor placement.
- `AHR09_GAMES=64` sets the fixed game count. Keep it unchanged on resume; use a new run ID to change configuration.

The code first verifies exact recovery for the selected architecture and devices, then starts or resumes. Games are played sequentially in this baseline; actor and learner placement on two GPUs does not imply simultaneous or saturated GPU execution. The previous generation stays immutable; notebook 11 decides whether a candidate improves playing strength. A completed run exports a candidate; it does not promote itself.

In [ ]:
TRAINING_RESULT = None
MODE09 = os.environ.get("AHR09_MODE", "validate")
if MODE09 not in ("validate", "train"):
    raise ValueError("AHR09_MODE must be validate or train")
if MODE09 == "train":
    run_id = os.environ.get("AHR09_RUN_ID", "go-selfplay-09-v1")
    if Path(run_id).name != run_id or run_id in (".", ".."):
        raise ValueError("Use a simple run ID")
    committed = selfplay09_read_checkpoint(PROJECT / "checkpoints" / run_id)
    initial_model, initial_metadata = None, None
    if committed is not None:
        # Exact09 resume takes its initialization and settings from its own committed state.
        training_model_config = GoAHRConfig(**committed["model_config"])
        training_config = SelfPlayConfig(**committed["config"])
        for variable, field_name in (("AHR09_DEVICE", "device"), ("AHR09_ACTOR_DEVICE", "actor_device"), ("AHR09_GAMES", "games")):
            if variable in os.environ and str(os.environ[variable]) != str(getattr(training_config, field_name)):
                raise ValueError("Resume configuration changed: " + variable)
        selected_initial = os.environ.get("AHR09_INITIAL_RUN")
        if selected_initial and selected_initial != (committed["initialization"].get("checkpoint_metadata") or {}).get("run_id"):
            raise ValueError("Selected initialization differs from the existing09 run")
        if training_config.purpose != "go_training":
            raise ValueError("A synthetic validation run cannot be resumed as real training")
    else:
        default_device = "cuda:0" if torch.cuda.is_available() else "cpu"
        learner_device = os.environ.get("AHR09_DEVICE", default_device)
        actor_device = os.environ.get("AHR09_ACTOR_DEVICE", learner_device)
        initialization = os.environ.get("AHR09_INITIALIZATION", "checkpoint")
        if initialization == "checkpoint":
            initial_run = os.environ.get("AHR09_INITIAL_RUN")
            if not initial_run:
                raise ValueError("Select AHR09_INITIAL_RUN, or explicitly choose AHR09_INITIALIZATION=random")
            initial_model, initial_metadata = selfplay09_generation_source(PROJECT, initial_run, device=learner_device,
                                                                   selection=os.environ.get("AHR09_INITIAL_SELECTION", "latest"))
            training_model_config = initial_model.config
        elif initialization == "random":
            training_model_config = GoAHRConfig(board_size=9)
        else:
            raise ValueError("AHR09_INITIALIZATION must be checkpoint or random")
        training_config = SelfPlayConfig(device=learner_device, actor_device=actor_device,
                                        games=int(os.environ.get("AHR09_GAMES", "64")))
    receipt_path = PROJECT / "reports/09_recovery_receipts" / (selfplay09_receipt_key(PROJECT, training_model_config, training_config) + ".json")
    if not receipt_path.is_file():
        validate_selfplay_recovery(PROJECT, training_model_config, device=training_config.device,
                                   actor_device=training_config.actor_device, production_config=training_config)
    TRAINING_RESULT = run_selfplay_training(PROJECT, run_id, training_model_config, training_config,
                                           initial_model=initial_model, initial_metadata=initial_metadata,
                                           allow_long_run=True)
    print(json.dumps({"phase": TRAINING_RESULT["phase"], "optimizer_step": TRAINING_RESULT["optimizer_step"],
                      "completed_games": TRAINING_RESULT["metrics"]["completed_games"],
                      "unknown_games": TRAINING_RESULT["metrics"]["unknown_games"],
                      "candidate_only": True}, indent=2))
else:
    print("VALIDATION_ONLY: full self-play training remains explicit; no strength claim or automatic promotion.")

## Saved result and handoff

Read the compact report below for the executed result. Notebook 11 should compare an actual training candidate with its exact initialization and earlier versions at matched thinking budgets. Synthetic recovery checkpoints are rejected by the shared inference loader's normal training-checkpoint path. No playing-strength result follows from successful loss computation or exact recovery.

In [ ]:
REPORT09 = {"schema": "ahr_notebook09_validation_v1", "status": "PASS",
            "default_mode": MODE09, "recovery": RECOVERY_REPORTS, "warmup_handoff": HANDOFF09,
            "training_optimizer_steps": 0 if TRAINING_RESULT is None else TRAINING_RESULT["optimizer_step"],
            "full_selfplay_started": TRAINING_RESULT is not None,
            "candidate_promotion": "NONE: requires notebook 11 arena",
            "claims": {"legal_transition_verification": True, "terminal_labels_only": True,
                       "strong_go_player": False, "learned_strength_improvement": "NOT_EVALUATED"},
            "presentation": {"saved_outputs": True, "html_preview": "reports/09_selfplay_preview.html",
                             "graphical_review": "NOT_PERFORMED: no graphical viewer in this execution environment"}}
atomic_json(PROJECT / "reports/09_selfplay_validation.json", REPORT09)
print(json.dumps({"status": REPORT09["status"], "recovery_backends": len(RECOVERY_REPORTS),
                  "full_selfplay_started": REPORT09["full_selfplay_started"],
                  "strength_improvement": "NOT_EVALUATED"}, indent=2))